# Tab-feature pretext CNN (`tab_cnn.py`, Colab)
Predicts the **16 tabular features from the image** (2-Gaussian MDN per feature) instead of
redshift — a pretext task that distils catalog photometry/morphology into the 64-d embedding.
Judged downstream: HGB over [this embedding (+tab16)] → σMAD on the v4-5 val.

References to beat: bins_emb+tab16 = **0.01130**, bins+mdn_emb+tab16 = **0.01127**.

Runtime → Change runtime type → **GPU**.


In [ ]:
import os
os.environ['CUTOUT_SIZE'] = '24'        # sample_v4.5 is 24x24
![ -d /content/Photo-Z-CNN-Model ] || git clone -b main https://github.com/zhhrozhh/Photo-Z-CNN-Model.git /content/Photo-Z-CNN-Model
%cd /content/Photo-Z-CNN-Model
!pip -q install mlflow


In [ ]:
# Google auth — if 'credential propagation was unsuccessful', just re-run this cell
from google.colab import auth; auth.authenticate_user()
!gcloud config set project macrocosm-lewagon -q


In [ ]:
!mkdir -p /content/data
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v4.5/catalog_v4.parquet /content/data/catalog_v1.parquet
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v4.5/images_*.npy /content/data/
DATA_DIR = '/content/data'

import tensorflow as tf
print('GPUs =', tf.config.list_physical_devices('GPU'))


In [ ]:
try:
    from google.colab import userdata
    MLFLOW_TOKEN = userdata.get('MLFLOW_TOKEN')
except Exception as e:
    print('no MLFLOW_TOKEN secret ->', e); MLFLOW_TOKEN = None


## Train
Targets are z-scored on the v4-5 train split (winsorized ±10σ), NaN-masked NLL, early-stop on val NLL.


In [ ]:
from tab_cnn import train_tab

hist, model, (f_mean, f_std) = train_tab(
    data_dir=DATA_DIR,
    crop=24, preproc='p99',
    N=None, batch=256, lr=3e-4, epochs=50,
    run_name='tab-mdn-v1',
    mlflow_token=MLFLOW_TOKEN,
)


## Downstream eval — the number that matters
Extract the 64-d embedding for all 600k, then HGB → σMAD on the v4-5 val (same protocol as every baseline).


In [ ]:
import glob, re
import numpy as np, pandas as pd, tensorflow as tf
import eval as ev, fusion as fu
from sklearn.ensemble import HistGradientBoostingRegressor

embn = tf.keras.Model(model.input, model.get_layer('embedding').output)
pp = ev.make_np_preprocess('p99')
paths = sorted(glob.glob(f'{DATA_DIR}/images_*.npy'), key=lambda p: int(re.findall(r'images_(\d+)_', p)[0]))
emb = np.concatenate([embn.predict(pp(np.asarray(np.load(p), 'float32')), batch_size=1024, verbose=0)
                      for p in paths])
np.save('/content/emb_tabmdn_v1.npy', emb); print('emb', emb.shape)

cat, z_all, o2i = fu.load_catalog_v4(DATA_DIR, 'catalog_v1.parquet')
X16, _ = fu.tabular_features(cat)
tr = np.array([o2i[int(o)] for o in pd.read_csv('splits/v4-5-train.csv').objid if int(o) in o2i])
va = np.array([o2i[int(o)] for o in pd.read_csv('splits/v4-5-validate.csv').objid if int(o) in o2i])
for name, X in {'tab_emb': emb, 'tab_emb+tab16': np.concatenate([emb, X16], 1)}.items():
    m = HistGradientBoostingRegressor(max_iter=4000, learning_rate=0.04, max_leaf_nodes=127,
        l2_regularization=1.0, early_stopping=True, validation_fraction=0.05,
        n_iter_no_change=40, random_state=0)
    m.fit(X[tr], np.log1p(z_all[tr]))
    zp = np.expm1(m.predict(X[va]))
    print(name, '| sigma_MAD', round(ev.sigma_mad(z_all[va], zp), 5),
          '| outlier', round(ev.outlier_rate(z_all[va], zp), 4))
# reference: bins_emb+tab16 = 0.01130 | bins+mdn_emb+tab16 = 0.01127


## Save the embedding (for combining with the other embeddings at home)


In [ ]:
!gcloud storage cp /content/emb_tabmdn_v1.npy gs://macrocosm-lewagon/results/
# model itself is already an MLflow artifact (tab-mdn-v1.keras) when logging is on
